In [1]:
import pandas as pd
import pandas as pd
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap, pmap
from libtpu.sdk import tpumonitoring
import math
import orbax.checkpoint as ocp
from functools import partial
import gc
gc.collect()

gold_second_futures = pd.read_csv('/content/GC2026_daily.csv')

gold_second_futures['time'] = pd.to_datetime(
    gold_second_futures['time']
)

gold_second_futures['timestamp'] = (
    gold_second_futures['time'].astype('int64') // 10**9
)

X = gold_second_futures[
    ["timestamp", "volume"]
].to_numpy()

y = gold_second_futures[
    ["close"]
].to_numpy()   # (N, 1)

train_data = int(len(X) * 0.8)

X_train = X[:train_data]
y_train = y[:train_data]

X_test = X[train_data:]
y_test = y[train_data:]


sequence_length = 10
batch_size = 32

def create_sequences(X, y, sequence_length):
    X_sequences = []
    y_sequences = []

    for i in range(len(X) - sequence_length + 1):
        X_sequences.append(
            X[i:i + sequence_length]
        )

        y_sequences.append(
            y[i:i + sequence_length]
        )

    return (
        jnp.array(X_sequences),
        jnp.array(y_sequences)
    )


X_train_seq, y_train_seq = create_sequences(
    X_train,
    y_train,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test,
    y_test,
    sequence_length
)




# 1. Grab your identified TPU core
tpu_device = jax.devices()[0]

# 2. Extract its current HBM allocator metrics
stats = tpu_device.memory_stats()

# 3. Calculate and print readable results
bytes_in_use = stats.get('bytes_in_use', 0)
peak_bytes = stats.get('peak_bytes_in_use', 0)

print(f"Device: {tpu_device}")
print(f"Current HBM in use: {bytes_in_use / (1024**2):.2f} MB")
print(f"Peak HBM recorded:  {peak_bytes / (1024**2):.2f} MB")

ModuleNotFoundError: No module named 'libtpu'

In [ ]:
# def adam_optimizer(curr_weights, curr_gradients, iter, layer, is_bias=False):

#     if is_bias:
#         moment_1_curr = (beta_1*(bias_moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
#         bias_moment_1_history[layer] = moment_1_curr
#     else:
#         moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
#         moment_1_history[layer] = moment_1_curr

#     hat_moment_1 = moment_1_curr/(1-beta_1**iter)

#     if is_bias:
#         moment_2_curr = beta_2*(bias_moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
#         bias_moment_2_history[layer] = moment_2_curr
#     else:
#         moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
#         moment_2_history[layer] = moment_2_curr

#     hat_moment_2 = moment_2_curr/(1-beta_2**iter)

#     return ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

In [ ]:
def single_tanh(act):
  res = (jnp.e**act - jnp.e**-act)/(jnp.e**act + jnp.e**-act)
  return res

single_tanh_jit = jit(single_tanh)


def tanh_activation(pre_activations):
  pre_act_shape = pre_activations.shape
  reshaped_pre_acts = jnp.reshape(pre_activations,(-1,))
  tanh_acts = vmap(single_tanh_jit)(reshaped_pre_acts)
  return jnp.reshape(tanh_acts,pre_act_shape)


In [ ]:
def meanAbsoluteLoss(predictions,true_labels):
  return jnp.mean(jnp.abs(predictions-true_labels))

def meanAbsoluteLossDerivation(predictions,true_labels):
  return jnp.mean(jnp.sign(predictions-true_labels))


In [ ]:
@jax.jit
def forward(layer_weights,inputs, biases):
  # layer_idx 0 for rnn layer 1, 1 for rnn layer 2, 2 for dense layer 1 which is also output layer
  sequence_length = inputs.shape[1]
  concatted_inputs = jnp.zeros((sequence_length,inputs.shape[0],66))
  layer_0_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[0].shape[1]))
  layer_1_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[1].shape[1]))
  layer_2_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[2].shape[1]))
  l0prev=jnp.zeros((32,64))
  l1prev=jnp.zeros((32,32))
  for i in range(sequence_length):
    print("seq_iter", i)
    inputs_for_layer_0 = jnp.concatenate([inputs[:,i,:],l0prev],axis=1)
    temp_layer_0_activation = tanh_activation((inputs_for_layer_0@layer_weights[0]) +biases[0])
    inputs_for_layer_1 = jnp.concatenate([temp_layer_0_activation,l1prev],axis=1)
    temp_layer_1_activation = tanh_activation((inputs_for_layer_1@layer_weights[1]) +biases[1])
    temp_layer_2_activation = (temp_layer_1_activation@layer_weights[2]) +biases[2]
    layer_0_activations.at[i].set(temp_layer_0_activation)
    layer_1_activations.at[i].set(temp_layer_1_activation)
    layer_2_activations.at[i].set(temp_layer_2_activation)
    concatted_inputs.at[i].set(inputs_for_layer_0)
    # breakpoint()
    l0prev,l1prev=temp_layer_0_activation,temp_layer_1_activation
  return concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations




In [ ]:
@partial(jax.jit, static_argnames=['sequence_length'])
def backward(
    layer_weights,
    sequence_length,
    concatted_inputs,
    layer_0_activations,
    layer_1_activations,
    layer_2_activations
    ):

  accumulated_layer_0_weights_changes = None
  accumulated_layer_1_weights_changes = None
  accumulated_layer_2_weights_changes = None

  accumulated_layer_0_bias_changes = None
  accumulated_layer_1_bias_changes = None
  accumulated_layer_2_bias_changes = None

  layer_0_next_timestep_delta = None
  layer_1_next_timestep_delta = None

  for i in range(sequence_length - 1, -1, -1):
    print("backprop_iter",i)
    preds = layer_2_activations[i]
    tanh_signs =  tanh_activation(preds)
    layer_2_weights_change = layer_1_activations[i-1] *  tanh_signs
    layer_2_bias_change = tanh_signs


    layer_1_activations_gradients = (tanh_signs *
            (
            layer_weights[2].T
            @ (1-layer_1_activations[i]**2)
            )
    ) # this is also the gradient for bias

    layer_1_weights_change = (
        layer_1_activations_gradients
            @ layer_0_activations[i-1]
          )

    layer_0_activations_gradients = (
        layer_1_activations_gradients
        *
            (
            layer_weights[2].T
            @ (1-layer_1_activations[i]**2)
            )
    ) # this is also the gradient for bias

    layer_0_weights_change = (
            layer_0_activations_gradients
            @ layer_0_activations[i-1]
          )
    if i < sequence_length-1:
      # here we have to handle the latent representation branch errors using BPTT
      input_cutoff_for_latent_repr_layer_1 =  layer_0_activations[i-1].shape[0] - 1
      temp_mask_for_rnn_layer_1_input = jnp.zeros((layer_0_activations[i-1].shape[0],layer_weights[1].shape[0]))

      row_indices = jnp.arange(layer_0_activations[i-1].shape[0])[:,jnp.newaxis]

      input_mask_for_rnn_layer_1 = jnp.where(row_indices>input_cutoff_for_latent_repr_layer_1,1,temp_mask_for_rnn_layer_1_input)


      masked_input = jnp.concatenate([layer_0_activations[i],layer_1_activations[i-1]],axis=1) * input_mask_for_rnn_layer_1

      print("masked_input ",masked_input)
      print("masked_input.T ",masked_input.T)

      weight_rows, weight_cols = jnp.indices(layer_weights[1].shape)
      weight_rows_cutoff = layer_weights[1].shape[0]//2
      temp_weights_mask_for_layer_1_recurrect_step = jnp.zeros(layer_weights[1].shape)
      weights_mask_for_layer_1_recurrect_step = jnp.where(weight_rows>weight_rows_cutoff,1,temp_weights_mask_for_layer_1_recurrect_step)

      masked_weights = layer_weights[1] * weights_mask_for_layer_1_recurrect_step

      print("layer_1_next_timestep_delta ",layer_1_next_timestep_delta)
      print("(1-layer_1_activations[i]**2) ",(1-layer_1_activations[i]**2))
      print("(masked_weights.T @ layer_1_next_timestep_delta) ",(masked_weights @ layer_1_next_timestep_delta))

      layer_1_recurrent_step_pre_act_gradients = (masked_weights @ layer_1_next_timestep_delta) * (1-layer_1_activations[i]**2)
      layer_1_recurrent_step_weights_change = layer_1_recurrent_step_pre_act_gradients @ masked_input

      layer_1_recurrent_step_bias_change = layer_1_recurrent_step_pre_act_gradients


      input_cutoff_for_latent_repr_layer_0 =  concatted_inputs[i-1].shape[0]//2
      temp_mask_for_rnn_layer_0 = jnp.zeros(concatted_inputs.shape)
      input_rows, input_cols = jnp.indices(temp_mask_for_rnn_layer_0.shape)
      input_mask_for_rnn_layer_0 = jnp.where(input_rows>input_cutoff_for_latent_repr_layer_0,1,temp_mask_for_rnn_layer_0)

      masked_input = concatted_inputs[i-1] * input_mask_for_rnn_layer_0

      weight_rows, weight_cols = jnp.indices(layer_weights[0].shape)
      weight_rows_cutoff = layer_weights[0].shape[0]//2
      temp_weights_mask_for_layer_0_recurrect_step = jnp.zeros(layer_weights[0].shape)
      weights_mask_for_layer_0_recurrect_step = jnp.where(weight_rows>weight_rows_cutoff,1,temp_weights_mask_for_layer_0_recurrect_step)

      masked_weights = layer_weights[0] * weights_mask_for_layer_0_recurrect_step


      layer_0_recurrent_step_pre_act_gradients = layer_0_next_timestep_delta * masked_weights * (1-layer_0_activations[i]**2)
      layer_0_recurrent_step_weights_change = layer_0_recurrent_step_pre_act_gradients @ masked_input

      layer_0_recurrent_step_bias_change = layer_0_recurrent_step_pre_act_gradients


    layer_0_next_timestep_delta = layer_0_activations_gradients
    layer_1_next_timestep_delta = layer_1_activations_gradients

  return (
      accumulated_layer_0_weights_changes,
      accumulated_layer_1_weights_changes,
      accumulated_layer_2_weights_changes,
      accumulated_layer_0_bias_changes,
      accumulated_layer_1_bias_changes,
      accumulated_layer_2_bias_changes
  )




In [ ]:
def he_initialization(layer_shapes):
    key = jax.random.key(1337)
    key, w_key = jax.random.split(key)
    layer_weights=[]
    layer_biases=[]

    # Define a He/Kaiming normal initializer (excellent for ReLU activations)
    initializer = jax.nn.initializers.he_normal()

    # Initialize the array
    for i, shape in enumerate(layer_shapes):
        layer_weights.append(initializer(w_key, shape, jnp.float32))
        layer_biases.append(jnp.zeros(shape[-1]))

    return layer_weights,layer_biases

In [ ]:
def training_loop():
  layer_weights_shapes = [(66,64),(96,32),(32,1)]
  layer_weights,layer_biases = he_initialization(layer_weights_shapes)

  batch_size = 32
  len_train_samples = len(X_train_seq)
  iters_per_epoch = math.ceil(len_train_samples / batch_size)
  epochs=10

  for epoch in range(epochs):
    for iter in range(iters_per_epoch):
      batch_start = iter * batch_size
      batch_end = (iter + 1) * batch_size
      batch_X = X_train_seq[batch_start:batch_end]
      batch_y = y_train_seq[batch_start:batch_end]
      concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations = forward(layer_weights,batch_X,layer_biases)

      # Fix: Transpose predictions from (Seq, Batch, 1) to (Batch, Seq, 1) to match batch_y
      preds_transposed = jnp.transpose(layer_2_activations, (1, 0, 2))
      loss = meanAbsoluteLoss(preds_transposed, batch_y)

      layer_0_weights_change, layer_1_weights_change, layer_2_weights_change, layer_0_bias_changes, layer_1_bias_changes, layer_2_bias_changes = backward(layer_weights,sequence_length,concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations)

  # Create a checkpoint manager or direct saver
  ackp = ocp.StandardCheckpointer()
  # Save parameters dictionary/tree to a path
  ackp.save('/content/rnn_weights', args=ocp.args.StandardSave((layer_weights,layer_biases)))

In [ ]:
training_loop()

seq_iter 0
seq_iter 1
seq_iter 2
seq_iter 3
seq_iter 4
seq_iter 5
seq_iter 6
seq_iter 7
seq_iter 8
seq_iter 9
backprop_iter 9
backprop_iter 8
masked_input  JitTracer<float32[32,96]>
masked_input.T  JitTracer<float32[96,32]>
layer_1_next_timestep_delta  JitTracer<float32[32,32]>
(1-layer_1_activations[i]**2)  JitTracer<float32[32,32]>
(masked_weights.T @ layer_1_next_timestep_delta)  JitTracer<float32[96,32]>


TypeError: dot_general requires contracting dimensions to have the same shape, got (96,) and (32,).